In [1]:
import keras
import keras_tuner as kt
from keras import layers

# Build model function for Keras Tuner
def build_model(hp):
    units = hp.Int(name="units", min_value=16, max_value=64, step=16) 
    model = keras.Sequential(
        [
            layers.Dense(units, activation="relu"),
            layers.Dense(10, activation="softmax"),
        ]
    )
    optimizer = hp.Choice(name="optimizer", values=["rmsprop", "adam"])
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model           

# keras tuner for hyperparameter optimization
class SimpleMLP(kt.HyperModel):
    def __init__(self, num_classes):                                  
        self.num_classes = num_classes

    def build(self, hp):                                              
        units = hp.Int(name="units", min_value=16, max_value=64, step=16)
        model = keras.Sequential(
            [
                layers.Dense(units, activation="relu"),
                layers.Dense(self.num_classes, activation="softmax"), 
            ]
        )
        optimizer = hp.Choice(name="optimizer", values=["rmsprop", "adam"])
        model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"],
        )
        return model   

hypermodel = SimpleMLP(num_classes=10)

# tuner = kt.BayesianOptimization(
#     hypermodel,                                                       
tuner = kt.BayesianOptimization(
    build_model,                                                       
    objective="val_accuracy",                                         
    max_trials=20,                                                     
    executions_per_trial=2,                                           
    directory="mnist_kt_test",                                        
    overwrite=True,                                                    
)

# Commented out the search space summary to avoid cluttering the output
tuner.search_space_summary()

# objective minimization or maximization
objective = kt.Objective(
    name="val_accuracy",                                          
    direction="max",                                              
)
tuner = kt.BayesianOptimization(
    build_model,
    objective=objective
)

# Test the tuner with a small dataset (e.g., MNIST) to ensure it works correctly
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.reshape((-1, 28 * 28)).astype("float32") / 255
x_test = x_test.reshape((-1, 28 * 28)).astype("float32") / 255
x_train_full = x_train[:]                                             
y_train_full = y_train[:]                                              
num_val_samples = 10000                                               
x_train, x_val = x_train[:-num_val_samples], x_train[-num_val_samples:] 
y_train, y_val = y_train[:-num_val_samples], y_train[-num_val_samples:] 
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5),    
]
tuner.search(                                                          
    x_train,
    y_train,
    batch_size=128,
    epochs=100,                                                    
    validation_data=(x_val, y_val),    
    callbacks=callbacks,
    verbose=2
)

# Get the best hyperparameters and model
top_n = 4
best_hps = tuner.get_best_hyperparameters(top_n)      

# Get the best epoch for the best hyperparameters
def get_best_epoch(hp):
    model = build_model(hp)
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", mode="min", patience=10               
        )                                                              
    ]
    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=100,
        batch_size=128,
        callbacks=callbacks,
    )
    val_loss_per_epoch = history.history["val_loss"]
    best_epoch = val_loss_per_epoch.index(min(val_loss_per_epoch)) + 1
    print(f"Best epoch: {best_epoch}")
    return best_epoch

# Get the best trained model for each set of hyperparameters
def get_best_trained_model(hp):
    best_epoch = get_best_epoch(hp)
    model = build_model(hp)
    model.fit(
        x_train_full, y_train_full, batch_size=128, epochs=int(best_epoch * 1.2)
    )
    return model

best_models = []
for hp in best_hps:
    model = get_best_trained_model(hp)
    model.evaluate(x_test, y_test)
    best_models.append(model)

best_models = tuner.get_best_models(top_n)

Trial 10 Complete [00h 00m 15s]
val_accuracy: 0.975600004196167

Best val_accuracy So Far: 0.975600004196167
Total elapsed time: 00h 16m 56s
Epoch 1/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8854 - loss: 0.4324 - val_accuracy: 0.9309 - val_loss: 0.2355
Epoch 2/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9379 - loss: 0.2204 - val_accuracy: 0.9464 - val_loss: 0.1848
Epoch 3/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9504 - loss: 0.1725 - val_accuracy: 0.9556 - val_loss: 0.1576
Epoch 4/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9592 - loss: 0.1428 - val_accuracy: 0.9606 - val_loss: 0.1414
Epoch 5/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9652 - loss: 0.1221 - val_accuracy: 0.9660 - val_loss: 0.1211
Epoch 6/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9694 - loss: 0.1061 - val_accuracy: 0.9684 - val_loss: 0.1122
Epoch 7/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9733 - loss: 0.093

c:\Users\Student\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(store)
c:\Users\Student\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(store)
